# Notebook 02: Introduction to Smart Contracts and Vyper

In this notebook, we move from the foundational concepts of blockchains to **Smart Contracts**. 

A Smart Contract is essentially code that lives on the blockchain. It can hold funds, receive data, make decisions, and execute logic exactly as programmed, without a middleman.

We will be learning **Vyper**, a Pythonic smart contract language for the Ethereum Virtual Machine (EVM).

We will cover:
1. Compiling Vyper in Colab.
2. Vyper Data Types.
3. Control Structures (Loops and If/Else).
4. Events and Logging.

### Prerequisites
We will use a library called `titanoboa` to compile and interact with Vyper code directly in memory inside this Python notebook. This saves us from having to set up a full external development environment just yet.

In [ ]:
!pip install titanoboa

## 1. Vyper Data Types

Vyper is strongly typed, meaning you must declare what kind of data a variable will hold. Let's look at a contract that simply declares various state variables (variables stored permanently on the blockchain).

In [ ]:
import boa

data_types_contract = """
# @version ^0.3.0

life_is_beautiful: public(bool)
var_int256: public(int256)
var_uint256: public(uint256)
my_grandma_wallet: public(address)
author: public(String[100])

# Mappings are like Python dictionaries
donaturs: public(HashMap[address, uint256])

@external
def __init__():
    self.life_is_beautiful = True
    self.var_int256 = -100
    self.var_uint256 = 100
    self.author = "Nelson"
"""

# Compile and deploy the contract to our local memory blockchain
contract = boa.loads(data_types_contract)

print("Contract Deployed!")
print(f"Author: {contract.author()}")
print(f"Is Life Beautiful?: {contract.life_is_beautiful()}")

## 2. Control Structures

Smart contracts can contain logic like `for` loops and `if/elif/else` statements. Because running code on the blockchain costs money (Gas), loops in Vyper are heavily restricted (they must have a fixed upper bound).

Let's see an example.

In [ ]:
control_structures_contract = """
# @version ^0.3.0

@external
@pure
def sum() -> int128:
    s: int128 = 0
    for i in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]:
        s += i
    return s

@external
@pure
def greet(time: String[10]) -> String[20]:
    if time == "morning":
        return "Good morning!"
    elif time == "evening":
        return "Good evening!"
    else:
        return "How are you?"
"""

contract2 = boa.loads(control_structures_contract)

print(f"Sum of 1-10: {contract2.sum()}")
print(f"Greeting (morning): {contract2.greet('morning')}")
print(f"Greeting (afternoon): {contract2.greet('afternoon')}")

## 3. Events and Logging

Smart contracts cannot easily "print" things to the console for the outside world to read. Instead, they emit **Events**. Applications (like web interfaces) can listen for these events.

Let's create a contract that accepts Ethereum (simulated) and logs a `Donation` event.

In [ ]:
events_contract = """
# @version ^0.3.0

event Donation:
    donatur: indexed(address)
    amount: uint256

@external
@payable
def donate():
    log Donation(msg.sender, msg.value)
"""

contract3 = boa.loads(events_contract)

# We will simulate sending 10^18 Wei (which is 1 Ether) to the contract
contract3.donate(value=10**18)

# Let's inspect the logs generated by our transaction
logs = contract3.get_logs()
print("Logs Emitted:", logs)